Python Tutorial - GeeksforGeeks: The Ultimate Spoon-Feeding of Python

In [9]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from bs4 import BeautifulSoup
import requests

In [10]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [11]:
openai = OpenAI()

In [12]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a senior technical content analyst with a sarcastic, witty tone.

You analyze web pages from learning platforms like GeeksforGeeks, especially programming tutorials.

Your job:
- Ignore navigation menus, ads, footer, and unrelated boilerplate text.
- Focus ONLY on actual educational content (tutorials, explanations, examples, headings).
- Identify the structure of the article (sections, topics, subtopics).
- Provide a detailed but engaging summary.

Style:
- Slightly snarky and humorous, but not excessive.
- Still informative and complete.
- Use markdown formatting (headings + bullet points where useful).
- Do NOT wrap output in code blocks.
"""

In [13]:
# Define our user prompt

user_prompt_prefix = """
You are given raw text extracted from a GeeksforGeeks page.

Task:
1. Identify the topic of the page.
2. Extract key sections covered in the tutorial.
3. Summarize each section briefly but clearly.
4. Mention practical concepts (code, DS, algorithms, etc.).
5. End with a short witty one-liner summary.

Important:
- Ignore navigation, ads, unrelated links, and repeated headers.
- Do not copy text verbatim.
- Be structured and moderately detailed (not one paragraph).

"""

In [14]:
# Scraper to fetch Python tutorial from GeeksforGeeks website

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/117.0.0.0 Safari/537.36"
    )
}


def fetch_website_contents(url, max_chars=3000):
    """
    Fetch and clean webpage text content.
    """

    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, "html.parser")

        # Extract title
        title = soup.title.string.strip() if soup.title else "No title"

        # Remove unwanted tags
        for tag in soup([
            "script",
            "style",
            "img",
            "svg",
            "noscript",
            "footer",
            "header",
            "nav",
            "aside",
            "form",
            "button",
        ]):
            tag.decompose()

        # Extract main text
        text = soup.get_text(separator="\n", strip=True)

        # Remove empty lines
        lines = [line.strip() for line in text.splitlines()]
        cleaned_text = "\n".join(line for line in lines if line)

        final_content = f"TITLE:\n{title}\n\nCONTENT:\n{cleaned_text}"

        return final_content[:max_chars]

    except requests.exceptions.RequestException as e:
        return f"Error fetching website: {e}"

In [15]:


def build_messages(website_content):
    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": f"{user_prompt_prefix}\n\n{website_content}"
        }
    ]

def summarize(url, model="gpt-4.1-mini"):
    # Fetch website content
    website_content = fetch_website_contents(url)

    # Create messages payload
    messages = build_messages(website_content)

    # Call OpenAI API
    response = openai.chat.completions.create(
        model=model,
        messages=messages
    )

    # Return summary text
    return response.choices[0].message.content


def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))


# Example usage
display_summary("https://www.geeksforgeeks.org/python-programming-language-tutorial/")

# Analysis of the GeeksforGeeks Python Tutorial Page

## 1. Topic of the Page:
This is a comprehensive **Python programming tutorial** from GeeksforGeeks aimed at beginners and intermediate learners. It covers everything from installing Python, writing your first "Hello, World!" program, to advanced concepts like decorators and object-oriented programming (OOP).

---

## 2. Key Sections Covered:
- Introduction and Motivation for Python
- Basics of Python Programming
- Functions in Python
- Data Structures in Python
- Object-Oriented Programming (OOP) Concepts

---

## 3. Section Summaries:

### Introduction and Motivation for Python
- Brief on why Python is popular: simple syntax, readability, rich libraries.
- Python as a high-level, cross-platform language used in various domains: AI, web development, data science.
- Prominent users include Google, Netflix, NASA.
- High demand for Python skills in the job market.
- A simple "Hello, World!" code snippet showcases the language's beginner-friendly nature.

### Basics of Python Programming
- Instructions on installing Python on your system (Windows, Mac, Linux).
- Fundamental concepts such as:
  - Writing your first program.
  - Understanding input/output operations.
  - Variables, data types, keywords, and operators.
  - Control flow: conditional statements and loops.
- Sets the foundation for further learning by clarifying essential syntax and programming logic.

### Functions in Python
- Comprehensive dive into Python's function syntax and usage.
- Understanding parameters, return values, and the scope of variables (local vs global).
- Special function features:
  - Recursion (functions calling themselves).
  - Variable-length arguments (*args and **kwargs).
  - First-class functions meaning you can pass functions as arguments and return them.
  - Lambda expressions for creating anonymous functions on the fly.
  - Built-in higher-order functions: map, filter, reduce which help in functional programming style.
  - Inner functions and decorators to modify behavior of other functions dynamically.

### Data Structures in Python
- Detailed exploration of Python’s built-in data types:
  - Strings, Lists, Tuples, Sets, Dictionaries, Arrays.
  - List comprehension—a Pythonic way to create and manipulate lists efficiently.
- Usage of Python's `collections` module for advanced structures such as:
  - Counters (for counting hashable objects).
  - Heapq (priority queues).
  - Deque (double-ended queues).
  - OrderedDict (dictionaries preserving insertion order).
  - Defaultdict (dictionary with default values for missing keys).
- Bridges basic data types to their practical applications in algorithms and data manipulation.

### Object-Oriented Programming (OOP) Concepts
- Introduction to core OOP principles:
  - Encapsulation: bundling data and methods.
  - Inheritance: reusing and extending classes.
  - Polymorphism: designing functions or methods to work on different types.
  - Abstract classes for blueprint classes.
  - Iterators for traversing collection objects.
- Practical understanding of:
  - Defining classes and creating objects.
  - The role of the `self` keyword as the instance reference.
- Helps build modular, reusable, and scalable Python applications.

---

## 4. Practical Concepts Included:
- Sample Python programs and syntax demonstrations.
- Control structures (if, else, loops).
- Function definitions and advanced functional programming features.
- Use of core and extended data types, including those from the `collections` module.
- Understanding and applying OOP principles for real-world code structuring.
- Mentions relevant libraries and frameworks relevant for fields like AI (TensorFlow, Scikit-learn) and web development (Django, Flask).

---

## 5. Witty One-liner Summary:
If Python were a Swiss Army knife, this tutorial would be your handy instruction manual—minus the blade injuries, but with all the coding cuts you need to slice through any problem.

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

In [16]:
# check if ollama is running
import requests
requests.get("http://localhost:11434").content

b'Ollama is running'

In [17]:
# download ollama 3.2:1b locally
!ollama pull llama3.2:1b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 74701a8c35f6: 100% ▕██████████████████▏ 1.3 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 4f659a1e86d7: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [19]:

ollama_system_prompt = """
You are an expert technical educator and content analyst.

Your job is to transform raw tutorial content into a structured, easy-to-understand learning guide.

Rules:

* Ignore navigation menus, ads, footers, sidebars, and unrelated content.
* Focus only on educational material.
* Identify the main topic and key concepts.
* Explain concepts in clear, developer-friendly language.
* Summarize code examples rather than reproducing them.
* Be informative, concise, and well-structured.
* Use markdown formatting.
* Do not wrap the response in code blocks.

Output Format:

# Topic

## Overview

Brief description of the tutorial.

## Key Concepts

Important topics covered.

## Section Breakdown

Short explanation of major sections.

## Practical Takeaways

Real-world applications and best practices.

## TL;DR

2-3 sentence summary.
"""

ollama_user_prompt_prefix = """
The following text was extracted from a technical tutorial webpage.

Your task is to create a concise study guide.

Requirements:

Identify the main topic.
Remove website noise and boilerplate content.
Summarize important concepts and sections.
Explain practical use cases.
Highlight tips, best practices, or interview-relevant points.
If algorithms or data structures are mentioned, briefly explain their purpose.

Create a summary detailed enough that someone can understand the tutorial without reading the original page.

Content:
"""


In [20]:
from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

def build_ollama_messages(website_content):
    return [
        {
            "role": "system",
            "content": ollama_system_prompt
        },
        {
            "role": "user",
            "content": f"{ollama_user_prompt_prefix}\n\n{website_content}"
        }
    ]

def summarize_ollama(url, model="llama3.2:1b"):
    # Fetch website content
    website_content = fetch_website_contents(url)

    # Create messages payload
    messages = build_ollama_messages(website_content)

    # Call OpenAI API
    response = ollama.chat.completions.create(
    model="llama3.2:1b", 
    messages=messages
    )

    # Return summary text
    return response.choices[0].message.content


def display_ollama_summary(url):
    summary = summarize_ollama(url)
    display(Markdown(summary))


# Example usage
display_ollama_summary("https://www.geeksforgeeks.org/python-programming-language-tutorial/")

**Python Tutorial - GeeksforGeeks**

**Overview**
===============

Python is a high-level scripting language used for data science, automation, artificial intelligence, web development, and more. It's easy to learn, readable, and has a strong library support, making it a popular choice among developers.

**Key Concepts**
================

* High-level language with clean syntax
* Popular for data science, automation, AI, web development, and more
* Provides libraries and frameworks like Django and Flask for web development and tools like Pandas, TensorFlow, and Scikit-learn for artificial intelligence, machine learning, and data analysis
* Cross-platform and compatible with Windows, Mac, and Linux without major changes

**Section Breakdown**
==================

* Installation: Installing Python on the system
* Basics: Understanding Python fundamentals, variables, operators, keywords, data types, conditional statements, loops, functions
* Data Structures: Strings, lists, tuples, dictionaries, sets, arrays, data structure basics
* OOP Concepts: Object-oriented programming (OOP) principles, classes and objects, inheritance, polymorphism

**Practical Takeaways**
=====================

* Use Python for data science, automation, and web development projects
* Implement basic data structures like lists, tuples, dictionaries, sets, arrays
* Write functions using Python's built-in functions, parameters, return values, variable scope, and more
* Apply OOP principles to build modular, reusable, and scalable code

**TL;DR**
===========

Python is a versatile programming language ideal for data science, automation, artificial intelligence, web development, and more. It provides a strong library support, cross-platform compatibility, and ease of use. By learning Python, you can implement basic data structures, understand OOP concepts, and write functions to accomplish various tasks.

**Important Algorithms and Data Structures**
=====================================================

* Lists: Mutable, ordered append/retrieve operations
* Tuples: Immutable, ordered append/retrieve operations
* Strings: Immobile, unordered string manipulation
* Objects (Dictionaries): Key-value pairs can be mutable or immutable